# Count lines in files

Utility to count lines of text in files under a specified folder path. Adjust `file_exts` to limit file types.

In [ ]:
from lib.count_lines import count_lines_in_folder
from pathlib import Path
from collections import defaultdict
import pandas as pd
import os
import re

# Folder sumber (raw) dan hasil cleaning
folder_raw = "../data/actual_raw_html"
folder_cleaned = "../data/cleaned_html_2"

# Override pasangan file untuk kasus nama yang tidak bisa diselesaikan lewat normalisasi umum
PAIR_OVERRIDES = {
    "guide.html": "customer_quick_guide.html",
    "profile_1.html": "customer_user_profile.html",
    "employee_1.html": "customer_export_employee.html",
    "report_gps_tracking.html": "customer_report_gps_tracking.html"
}

def count_chars_in_folder(folder_path):
    """Count total characters for each HTML file in a folder."""
    char_counts = {}
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if os.path.isfile(file_path) and filename.endswith(".html"):
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        char_counts[file_path] = len(f.read())
                except Exception:
                    pass
    return char_counts

def stem_no_ext(filename: str) -> str:
    return Path(filename).stem

def logical_key_raw(filename: str) -> str:
    """actual_raw_html naming key (preserve full stem to avoid ambiguous collapse)."""
    return stem_no_ext(filename)

def logical_key_cleaned(filename: str) -> str:
    """cleaned_html_2 naming key.
    Example: customer_report_scan_gps -> scan_gps
             customer_setting_profile -> profile
             customer_dashboard -> dashboard
    """
    stem = stem_no_ext(filename)
    if stem.startswith("customer_"):
        stem = stem[len("customer_"):]

    parts = stem.split("_")
    if parts and parts[0] in {"report", "setting", "export", "office", "employee", "device", "attendance", "organization"}:
        if len(parts) > 1:
            stem = "_".join(parts[1:])
    return stem

def df_to_markdown(df: pd.DataFrame) -> str:
    """Convert DataFrame to Markdown table without external dependencies."""
    if df.empty:
        return "_Tidak ada data yang dapat ditampilkan._"
    headers = [str(c) for c in df.columns]
    lines = []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in df.iterrows():
        vals = [str(row[c]).replace("\n", " ") for c in df.columns]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

# Hitung lines
counts_lines_raw, _ = count_lines_in_folder(folder_raw)
counts_lines_cleaned, _ = count_lines_in_folder(folder_cleaned)

# Hitung characters
counts_chars_raw = count_chars_in_folder(folder_raw)
counts_chars_cleaned = count_chars_in_folder(folder_cleaned)

# Mapping basename agar konsisten
lines_raw_by_name = {os.path.basename(p): c for p, c in counts_lines_raw.items()}
lines_cleaned_by_name = {os.path.basename(p): c for p, c in counts_lines_cleaned.items()}
chars_raw_by_name = {os.path.basename(p): c for p, c in counts_chars_raw.items()}
chars_cleaned_by_name = {os.path.basename(p): c for p, c in counts_chars_cleaned.items()}

# Group file berdasarkan key logis
raw_groups = defaultdict(list)
for fname in lines_raw_by_name:
    if fname in chars_raw_by_name:
        raw_groups[logical_key_raw(fname)].append(fname)

cleaned_groups = defaultdict(list)
for fname in lines_cleaned_by_name:
    if fname in chars_cleaned_by_name:
        cleaned_groups[logical_key_cleaned(fname)].append(fname)

for key in raw_groups:
    raw_groups[key] = sorted(raw_groups[key])
for key in cleaned_groups:
    cleaned_groups[key] = sorted(cleaned_groups[key])

# Pair dari override dulu agar tidak bentrok dengan pairing otomatis
pairs = []
used_raw = set()
used_cleaned = set()

for raw_fname, cleaned_fname in PAIR_OVERRIDES.items():
    if (
        raw_fname in lines_raw_by_name
        and raw_fname in chars_raw_by_name
        and cleaned_fname in lines_cleaned_by_name
        and cleaned_fname in chars_cleaned_by_name
    ):
        pairs.append((raw_fname, cleaned_fname, "override"))
        used_raw.add(raw_fname)
        used_cleaned.add(cleaned_fname)

# Pair otomatis berdasarkan key normalisasi
all_keys = sorted(set(raw_groups.keys()) & set(cleaned_groups.keys()))
for key in all_keys:
    raw_candidates = [f for f in raw_groups[key] if f not in used_raw]
    cleaned_candidates = [f for f in cleaned_groups[key] if f not in used_cleaned]
    pair_count = min(len(raw_candidates), len(cleaned_candidates))
    for i in range(pair_count):
        raw_fname = raw_candidates[i]
        cleaned_fname = cleaned_candidates[i]
        pairs.append((raw_fname, cleaned_fname, key))
        used_raw.add(raw_fname)
        used_cleaned.add(cleaned_fname)

# Debug info
print(f"Raw HTML files: {len(lines_raw_by_name)}")
print(f"Cleaned HTML files: {len(lines_cleaned_by_name)}")
print(f"Logical keys matched: {len(all_keys)}")
print(f"Override pairs used: {sum(1 for _,_,k in pairs if k == 'override')}")
print(f"File pairs compared: {len(pairs)}")

# Tabel hasil per file
rows = []
for raw_fname, cleaned_fname, match_key in pairs:
    raw_lines = lines_raw_by_name[raw_fname]
    cleaned_lines = lines_cleaned_by_name[cleaned_fname]
    raw_chars = chars_raw_by_name[raw_fname]
    cleaned_chars = chars_cleaned_by_name[cleaned_fname]

    lines_removed = raw_lines - cleaned_lines
    chars_removed = raw_chars - cleaned_chars

    lines_decrease_pct = (lines_removed / raw_lines * 100) if raw_lines else 0
    chars_decrease_pct = (chars_removed / raw_chars * 100) if raw_chars else 0

    rows.append({
        "match_key": match_key,
        "raw_filename": raw_fname,
        "cleaned_filename": cleaned_fname,
        "raw_lines": raw_lines,
        "cleaned_lines": cleaned_lines,
        "lines_removed": lines_removed,
        "lines_decrease_percent": round(lines_decrease_pct, 2),
        "raw_chars": raw_chars,
        "cleaned_chars": cleaned_chars,
        "chars_removed": chars_removed,
        "chars_decrease_percent": round(chars_decrease_pct, 2),
    })

if rows:
    df_reduction = pd.DataFrame(rows).sort_values("lines_decrease_percent", ascending=False).reset_index(drop=True)
    df_avg = pd.DataFrame([
        {
            "avg_lines_removed": round(df_reduction["lines_removed"].mean(), 2),
            "avg_lines_decrease_percent": round(df_reduction["lines_decrease_percent"].mean(), 2),
            "avg_chars_removed": round(df_reduction["chars_removed"].mean(), 2),
            "avg_chars_decrease_percent": round(df_reduction["chars_decrease_percent"].mean(), 2),
            "total_files_compared": len(df_reduction),
        }
    ])
else:
    df_reduction = pd.DataFrame(columns=[
        "match_key",
        "raw_filename",
        "cleaned_filename",
        "raw_lines",
        "cleaned_lines",
        "lines_removed",
        "lines_decrease_percent",
        "raw_chars",
        "cleaned_chars",
        "chars_removed",
        "chars_decrease_percent",
    ])
    df_avg = pd.DataFrame([
        {
            "avg_lines_removed": 0,
            "avg_lines_decrease_percent": 0,
            "avg_chars_removed": 0,
            "avg_chars_decrease_percent": 0,
            "total_files_compared": 0,
        }
    ])

display(df_reduction)
display(df_avg)

# Unmatched diagnostics
unmatched_raw = sorted(set(lines_raw_by_name.keys()) - used_raw)
unmatched_cleaned = sorted(set(lines_cleaned_by_name.keys()) - used_cleaned)
print("\nUnmatched raw files:", unmatched_raw)
print("Unmatched cleaned files:", unmatched_cleaned)

# Export tabel ke Markdown
output_md = Path("./line_char_reduction_summary.md")
content_md = []
content_md.append("# Ringkasan Penurunan HTML Setelah Cleaning")
content_md.append("")
content_md.append(f"- Folder raw: `{folder_raw}`")
content_md.append(f"- Folder cleaned: `{folder_cleaned}`")
content_md.append(f"- Total pasangan file dibandingkan: **{len(pairs)}**")
content_md.append("")
content_md.append("## Tabel Per File")
content_md.append(df_to_markdown(df_reduction))
content_md.append("")
content_md.append("## Rata-Rata Penurunan")
content_md.append(df_to_markdown(df_avg))
content_md.append("")

output_md.write_text("\n".join(content_md), encoding="utf-8")
print(f"Markdown exported: {output_md.resolve()}")

Raw HTML files: 47
Cleaned HTML files: 47
Logical keys matched: 43
Override pairs used: 4
File pairs compared: 47


,match_key,raw_filename,cleaned_filename,raw_lines,cleaned_lines,lines_removed,lines_decrease_percent,raw_chars,cleaned_chars,chars_removed,chars_decrease_percent
0,override,employee_1.html,customer_export_employee.html,6132,88,6044,98.56,260511,2886,257625,98.89
1,override,profile_1.html,customer_user_profile.html,6116,102,6014,98.33,260750,4126,256624,98.42
2,attlog,attlog.html,customer_export_attlog.html,6229,141,6088,97.74,264652,5021,259631,98.10
3,device_admin,device_admin.html,customer_office_device_admin.html,6563,161,6402,97.55,277951,5897,272054,97.88
4,attendance,attendance.html,customer_report_attendance.html,8743,246,8497,97.19,405617,10077,395540,97.52
5,no_scan,no_scan.html,customer_report_no_scan.html,6375,186,6189,97.08,273348,7669,265679,97.19
6,rekap_work_monitoring,rekap_work_monitoring.html,customer_report_rekap_work_monitoring.html,6419,207,6212,96.78,275443,8764,266679,96.82
7,overtime_request,overtime_request.html,customer_report_overtime_request.html,6656,226,6430,96.60,287105,10235,276870,96.44
8,leave_request,leave_request.html,customer_report_leave_request.html,6843,245,6598,96.42,294044,10962,283082,96.27
9,attendance_detail,attendance_detail.html,customer_report_attendance_detail.html,5973,242,5731,95.95,255980,10109,245871,96.05


,avg_lines_removed,avg_lines_decrease_percent,avg_chars_removed,avg_chars_decrease_percent,total_files_compared
0,7467.79,89.31,359821.43,89.11,47



Unmatched raw files: []
Unmatched cleaned files: []
Markdown exported: C:\Documents\Kuliah\Skripsi\in-app-navigational-agent\cleaning\line_char_reduction_summary.md


: 

In [5]:
from pathlib import Path

# Preview markdown export
md_path = Path("./line_char_reduction_summary.md")
if md_path.exists():
    preview = md_path.read_text(encoding="utf-8")
    print(preview[:2000])  # preview awal file
else:
    print("File markdown belum ada. Jalankan Cell 2 terlebih dahulu.")

# Ringkasan Penurunan HTML Setelah Cleaning

- Folder raw: `../data/actual_raw_html`
- Folder cleaned: `../data/cleaned_html_2`
- Total pasangan file dibandingkan: **44**

## Tabel Per File
| match_key | raw_filename | cleaned_filename | raw_lines | cleaned_lines | lines_removed | lines_decrease_percent | raw_chars | cleaned_chars | chars_removed | chars_decrease_percent |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| employee | employee_1.html | customer_export_employee.html | 6132 | 88 | 6044 | 98.56 | 260511 | 2886 | 257625 | 98.89 |
| attlog | attlog.html | customer_export_attlog.html | 6229 | 141 | 6088 | 97.74 | 264652 | 5021 | 259631 | 98.1 |
| device_admin | device_admin.html | customer_office_device_admin.html | 6563 | 161 | 6402 | 97.55 | 277951 | 5897 | 272054 | 97.88 |
| attendance | attendance.html | customer_report_attendance.html | 8743 | 246 | 8497 | 97.19 | 405617 | 10077 | 395540 | 97.52 |
| no_scan | no_scan.html | customer_report_no_scan.html

In [4]:
count_lines_in_folder("../data/json_html")

({'..\\data\\json_html\\107.155.65.156_81_customer_attendance_day_off.json': 136,
  '..\\data\\json_html\\107.155.65.156_81_customer_attendance_employee_schedule.json': 97,
  '..\\data\\json_html\\107.155.65.156_81_customer_attendance_leave.json': 729,
  '..\\data\\json_html\\107.155.65.156_81_customer_attendance_leave_allowance.json': 170,
  '..\\data\\json_html\\107.155.65.156_81_customer_attendance_overtime.json': 360,
  '..\\data\\json_html\\107.155.65.156_81_customer_attendance_schedule.json': 175,
  '..\\data\\json_html\\107.155.65.156_81_customer_attendance_status.json': 235,
  '..\\data\\json_html\\107.155.65.156_81_customer_cart.json': 845,
  '..\\data\\json_html\\107.155.65.156_81_customer_changelog.json': 527,
  '..\\data\\json_html\\107.155.65.156_81_customer_dashboard.json': 137,
  '..\\data\\json_html\\107.155.65.156_81_customer_device.json': 243,
  '..\\data\\json_html\\107.155.65.156_81_customer_device_api_sdk.json': 111,
  '..\\data\\json_html\\107.155.65.156_81_custom

In [6]:
import os
import pandas as pd

# Set these two filenames to compare one file from each folder.
# Example:
# scraped_file = "customer_attendance_leave.html"
# cleaned_file = "customer_attendance_leave.html"
scraped_file = "../data/cleaned_html/customer_attendance_leave.html"
cleaned_file = "../traversal/output/customer_attendance_leave_navigation_nodes.html"

# folder_scraped = "../data/scraped_html"
# folder_cleaned = "../data/cleaned_html"

# Support both plain filenames and full/relative file paths.
# If a path is provided, split it into folder + filename.
if scraped_file:
    scraped_file = os.path.normpath(scraped_file)
    if os.path.dirname(scraped_file):
        folder_scraped = os.path.dirname(scraped_file)
        scraped_file = os.path.basename(scraped_file)

if cleaned_file:
    cleaned_file = os.path.normpath(cleaned_file)
    if os.path.dirname(cleaned_file):
        folder_cleaned = os.path.dirname(cleaned_file)
        cleaned_file = os.path.basename(cleaned_file)
def _count_lines_in_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return len(f.read().splitlines())


def _count_chars_in_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return len(f.read())


def compare_two_html_files(scraped_filename, cleaned_filename, scraped_folder, cleaned_folder):
    scraped_path = os.path.join(scraped_folder, scraped_filename)
    cleaned_path = os.path.join(cleaned_folder, cleaned_filename)

    scraped_lines = _count_lines_in_file(scraped_path)
    cleaned_lines = _count_lines_in_file(cleaned_path)
    scraped_chars = _count_chars_in_file(scraped_path)
    cleaned_chars = _count_chars_in_file(cleaned_path)

    lines_removed = scraped_lines - cleaned_lines
    chars_removed = scraped_chars - cleaned_chars

    lines_decrease_percent = (lines_removed / scraped_lines * 100) if scraped_lines else 0
    chars_decrease_percent = (chars_removed / scraped_chars * 100) if scraped_chars else 0

    df = pd.DataFrame([
        {
            'scraped_filename': scraped_filename,
            'cleaned_filename': cleaned_filename,
            'scraped_html_lines': scraped_lines,
            'cleaned_html_lines': cleaned_lines,
            'lines_removed': lines_removed,
            'lines_decrease_percent': round(lines_decrease_percent, 2),
            'scraped_html_chars': scraped_chars,
            'cleaned_html_chars': cleaned_chars,
            'chars_removed': chars_removed,
            'chars_decrease_percent': round(chars_decrease_percent, 2),
        }
    ])

    return df


if scraped_file and cleaned_file:
    display(compare_two_html_files(scraped_file, cleaned_file, folder_scraped, folder_cleaned))
else:
    print("Set scraped_file and cleaned_file first, then run this cell.")

,scraped_filename,cleaned_filename,scraped_html_lines,cleaned_html_lines,lines_removed,lines_decrease_percent,scraped_html_chars,cleaned_html_chars,chars_removed,chars_decrease_percent
0,customer_attendance_leave.html,customer_attendance_leave_navigation_nodes.html,4201,3370,831,19.78,379173,297742,81431,21.48
